In [1]:
import sys
sys.path.append("..")  #  the notebook runs inside notebook/, and src/ is one folder up

from src.normalize import norm_name

print(repr(norm_name("Récord Office (Pvt.) Ltd")))   # expect 'record office'
print(repr(norm_name("फर्स्ट फूड प्राइवेट लिमिटेड")))      # expect ''
print(repr(norm_name("FH  FH Business, LLC")))        # expect 'fh fh business'

'record office'
''
'fh fh business'


In [2]:
from src.normalize import norm_address

print(sorted(norm_address("88 OLIVE CIR, LEBANON, TN")))
print(norm_address("88 Olive Circle, Lebanon, TN") == norm_address("Lebanon, Tennessee, 88 Olive Cir"))
print(sorted(norm_address("CALUMET CITY, 351 HOXIE AVE, N/A, IL")))
print(repr(norm_address(float("nan"))))

['88', 'circle', 'lebanon', 'olive', 'tennessee']
True
['351', 'avenue', 'calumet', 'city', 'hoxie', 'illinois']
frozenset()


In [3]:
from src.blocking import name_keys

print(sorted(name_keys("Record Office Pvt Ltd", "US")))
print(sorted(name_keys("Rcdbr Offie", "US")))
print(sorted(name_keys("FH FH Business", "US")))
print(name_keys("राजस्थान स्टोर", "IN"))

['n|US|office', 'n|US|record', 'p|US|offi', 'p|US|reco']
['n|US|offie', 'n|US|rcdbr', 'p|US|offi', 'p|US|rcdb']
['n|US|business', 'p|US|busi']
set()


In [4]:
from src.blocking import address_keys

print(sorted(address_keys("CALUMET CITY, 351 HOXIE AVE, N/A, IL", "US")))
print(sorted(address_keys("88 OLIVE CIR, LEBANON, TN", "US")))
print(address_keys(float("nan"), "US"))
print(address_keys("Lebanon, Tennessee", "US"))

['a|US|351|avenue', 'a|US|351|calumet', 'a|US|351|city', 'a|US|351|hoxie', 'a|US|351|illinois']
['a|US|88|circle', 'a|US|88|lebanon', 'a|US|88|olive', 'a|US|88|tennessee']
set()
set()


In [10]:
import sys; sys.path.append("..")
import pandas as pd

%load_ext autoreload
# WHY: re-imports src/ after you edit blocking.py, so no kernel restart is needed
%autoreload 2

from src.blocking import build_index, block_sizes

# WHY: replace the file names below with the real ones from the dataset/dev listing
s2_dev = pd.read_csv("../dataset/dev/s2_dev.tsv", sep="\t")
s3_dev = pd.read_csv("../dataset/dev/s3_dev.tsv", sep="\t")

pool = pd.concat([s2_dev, s3_dev], ignore_index=True)
print(len(pool), pool.columns.tolist())   # expect 467494 rows

index = build_index(pool)
sizes = block_sizes(index)

print("number of keys:", len(sizes))
print(sizes.describe(percentiles=[0.5, 0.9, 0.99, 0.999]))
print(sizes.sort_values(ascending=False).head(15))

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
467494 ['entity_id', 'business_name', 'business_address', 'country']
number of keys: 1153212
count    1.153212e+06
mean     4.447934e+00
std      5.851868e+01
min      1.000000e+00
50%      1.000000e+00
90%      5.000000e+00
99%      3.600000e+01
99.9%    4.050000e+02
max      1.462400e+04
dtype: float64
p|US|cent           14624
n|US|center         13202
p|US|part           12537
n|US|partners       12249
n|US|com            11641
p|India|indi        11406
n|US|and            11031
n|India|india       10778
p|US|serv           10557
p|US|grou            9558
n|US|group           9548
p|India|serv         9437
n|US|care            7407
p|US|asso            7383
n|India|services     7299
dtype: int64
